# ChromaDB Multimodal Vector Indexing & Nearest Neighbor Retrieval
Demonstrates vector upsert, cosine distance computation, and metadata-filtered retrieval.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../../backend'))
from app.db.chroma_client import chroma_store
from app.rag.embeddings import embedding_service

# Index multiple enterprise entities
entities = [
    ("ETH-AIR", "Ethiopian Airlines operating modern Boeing and Airbus fleet out of Bole Addis Ababa", {"sector": "aviation"}),
    ("ETH-TEL", "Ethio Telecom deploying 5G networks and Telebirr digital payment platform", {"sector": "telecom"}),
    ("CBE-BANK", "Commercial Bank of Ethiopia headquarters tower in financial district Addis Ababa", {"sector": "banking"})
]

for doc_id, text, meta in entities:
    emb = embedding_service.embed_query(text)
    chroma_store.add_item(doc_id, emb, text, meta)

# Query vector store
query_text = "commercial aviation aircraft fleet"
q_emb = embedding_service.embed_query(query_text)
results = chroma_store.query(q_emb, n_results=2)

print(f"Indexed entities count: {chroma_store.count()}")
print(f"Query: '{query_text}'")
print("Top Retrieved Matches:")
for i, match_id in enumerate(results['ids'][0]):
    doc = results['documents'][0][i]
    meta = results['metadatas'][0][i]
    dist = results['distances'][0][i]
    print(f"  Rank #{i+1} [{match_id}] Distance: {dist:.4f} | Sector: {meta.get('sector')} | Content: {doc[:70]}...")


Indexed entities count: 4
Query: 'commercial aviation aircraft fleet'
Top Retrieved Matches:
  Rank #1 [ETH-AIR] Distance: 0.1245 | Sector: aviation | Content: Ethiopian Airlines operating modern Boeing and Airbus fleet out of Bole...
  Rank #2 [ETH-TEL] Distance: 0.6841 | Sector: telecom | Content: Ethio Telecom deploying 5G networks and Telebirr digital payment plat...
